# Chapter 2: Markov Processes — Python Examples

This notebook provides computational illustrations of the key concepts in Chapter 2.  
Sections follow the order of the chapter.

**Contents**
1. [Finite Markov Chains and the Transition Operator](#sec1)
2. [Stationary Distributions](#sec2)
3. [Eigenfunctions with Unit Eigenvalues and Invariant Events](#sec3)
4. [Ergodicity and the Resolvent Operator](#sec4)
5. [Periodicity and Skip-Sampling](#sec5)
6. [Strong Contraction and Spectral Radius](#sec6)
7. [Convergence of Multi-Period Forecasts (Stochastic Stability)](#sec7)
8. [Vector Autoregressions: Ergodic and Non-Ergodic](#sec8)
9. [Inventing a Past: Moving-Average Representation](#sec9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.linalg import solve_discrete_lyapunov, eig

rng = np.random.default_rng(42)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

---
<a id='sec1'></a>
## 1. Finite Markov Chains and the Transition Operator

A finite-state Markov chain is fully characterized by an $n \times n$ **transition matrix**
$\mathbb{P}$ whose $(i,j)$ entry is the probability of moving from state $i$ to state $j$
in one period.  Rows sum to one.

The **conditional expectation operator** $\mathbb{T}$ acts on a vector of function values
$\mathbf{f}$ via
$$
(\mathbb{T}\mathbf{f})_i = \sum_j p_{ij}\, f_j,
\qquad \text{i.e.,}\quad \mathbb{T}\mathbf{f} = \mathbb{P}\mathbf{f}.
$$
Iterating $j$ times gives $\mathbb{T}^j \mathbf{f} = \mathbb{P}^j \mathbf{f}$, the
$j$-step-ahead conditional expectation.

We work with the three illustrative chains from the chapter.

In [ ]:
# Example 2.4 (MC1): periodic chain  P = [[0,1],[1,0]]
P_mc1 = np.array([[0., 1.],
                  [1., 0.]])

# Example 2.5 (MC2): two absorbing states  P = I_2
P_mc2 = np.eye(2)

# Example 2.6 (MC3): one ergodic block + one absorbing state
P_mc3 = np.array([[0.8, 0.2, 0.0],
                  [0.1, 0.9, 0.0],
                  [0.0, 0.0, 1.0]])

def simulate_chain(P, init_state, T=200, rng=rng):
    """Simulate a finite Markov chain for T steps."""
    n = P.shape[0]
    states = np.empty(T, dtype=int)
    states[0] = init_state
    for t in range(T - 1):
        states[t + 1] = rng.choice(n, p=P[states[t]])
    return states

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
chains = [(P_mc1, 0, 'MC1 (periodic, 2 states)'),
          (P_mc2, 0, 'MC2 (two absorbing states)'),
          (P_mc3, 0, 'MC3 (one ergodic block + absorbing state)')]

for ax, (P, s0, title) in zip(axes, chains):
    path = simulate_chain(P, s0, T=200)
    ax.step(range(200), path, where='post', linewidth=0.8)
    ax.set_ylabel('State')
    ax.set_title(title)
    ax.set_yticks(range(P.shape[0]))

axes[-1].set_xlabel('Time $t$')
fig.suptitle('Sample paths of three illustrative Markov chains', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='sec2'></a>
## 2. Stationary Distributions

A **stationary distribution** $\mathbf{q}$ satisfies the left-eigenvector equation
$$
\mathbf{q}' \mathbb{P} = \mathbf{q}', \qquad \mathbf{q} \ge 0, \quad \sum_i q_i = 1.
$$
Equivalently, $\mathbf{q}$ is a left eigenvector of $\mathbb{P}$ associated with eigenvalue 1,
or a right eigenvector of $\mathbb{P}'$ associated with eigenvalue 1.

For chains with multiple stationary distributions (MC2 and MC3), the full set of stationary
distributions is the convex hull of the **ergodic building-block** $Q$'s.

In [ ]:
def stationary_distributions(P, tol=1e-10):
    """
    Return all stationary distributions as rows of a matrix.
    Each is a right eigenvector of P' with eigenvalue 1, normalized to sum to 1.
    """
    eigenvalues, eigenvectors = eig(P.T)
    unit_idx = np.where(np.abs(eigenvalues - 1.0) < tol)[0]
    stats = []
    for idx in unit_idx:
        v = eigenvectors[:, idx].real
        if np.all(v >= -tol):          # nonnegative
            v = np.maximum(v, 0.0)
            v /= v.sum()
            stats.append(v)
    return stats

for P, name in [(P_mc1, 'MC1'), (P_mc2, 'MC2'), (P_mc3, 'MC3')]:
    stats = stationary_distributions(P)
    print(f"{name}: {len(stats)} stationary distribution(s)")
    for q in stats:
        print(f"  q = {np.round(q, 4)}")

In [ ]:
# MC3: the convex hull of the two building-block Q's
# q(pi) = pi * [1/3, 2/3, 0] + (1-pi) * [0, 0, 1]  for pi in [0,1]
q1 = np.array([1/3, 2/3, 0.0])
q2 = np.array([0.0, 0.0, 1.0])

pis = np.linspace(0, 1, 200)
convex_hull = np.outer(pis, q1) + np.outer(1 - pis, q2)   # shape (200, 3)

fig, ax = plt.subplots(figsize=(8, 3.5))
for s, label in enumerate(['State 1', 'State 2', 'State 3']):
    ax.plot(pis, convex_hull[:, s], label=label)
ax.set_xlabel(r'$\pi$')
ax.set_ylabel('Stationary probability')
ax.set_title('MC3: convex hull of the two building-block stationary distributions')
ax.legend()
plt.tight_layout()
plt.show()

---
<a id='sec3'></a>
## 3. Eigenfunctions with Unit Eigenvalues and Invariant Events

A function $\mathbf{f}$ satisfying $\mathbb{T}\mathbf{f} = \mathbf{f}$ (i.e., $\mathbb{P}\mathbf{f} = \mathbf{f}$)
is a **right eigenvector** of $\mathbb{P}$ associated with eigenvalue 1.
By Proposition 2.1, $\{f(X_t)\}$ is constant over time with probability one.

The Markov process is **ergodic** with respect to $Q$ when every such eigenfunction
is constant $Q$-almost surely.  When non-constant eigenfunctions exist,
their level sets identify invariant events.

We verify this numerically for all three chains.

In [ ]:
def unit_eigenfunctions(P, tol=1e-10):
    """Return right eigenvectors of P associated with eigenvalue 1."""
    eigenvalues, eigenvectors = eig(P)
    unit_idx = np.where(np.abs(eigenvalues - 1.0) < tol)[0]
    return [(eigenvalues[i].real, eigenvectors[:, i].real) for i in unit_idx]

for P, name in [(P_mc1, 'MC1'), (P_mc2, 'MC2'), (P_mc3, 'MC3')]:
    fns = unit_eigenfunctions(P)
    print(f"\n{name}: right eigenvectors with eigenvalue 1")
    for lam, f in fns:
        f_norm = f / f[0] if np.abs(f[0]) > 1e-12 else f
        print(f"  f = {np.round(f_norm, 4)}")

In [ ]:
# Verify numerically: f(X_t) stays constant along a MC3 sample path
# Use the non-constant eigenfunction [1, 1, 0] (unnormalized) for the ergodic block vs state 3
# Actually: eigenfunction for MC3 that distinguishes blocks is [1, 1, 0] vs the constant one.

_, right_evecs = eig(P_mc3)
vals, _ = eig(P_mc3)
print("MC3 eigenvalues:", np.round(np.sort(vals.real)[::-1], 6))

# The two unit-eigenvalue eigenfunctions
fns_mc3 = unit_eigenfunctions(P_mc3)
f_nonconstant = fns_mc3[1][1]  # second one — varies across blocks
f_nonconstant /= np.max(np.abs(f_nonconstant))

path = simulate_chain(P_mc3, init_state=0, T=100)
f_along_path = f_nonconstant[path]

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].step(range(100), path, where='post', linewidth=0.8)
axes[0].set_ylabel('State')
axes[0].set_title('MC3: sample path (started in ergodic block)')
axes[1].step(range(100), f_along_path, where='post', color='C1', linewidth=0.8)
axes[1].set_ylabel(r'$f(X_t)$')
axes[1].set_title(r'Non-constant eigenfunction $f(X_t)$ — constant along any path')
axes[1].set_xlabel('Time $t$')
plt.tight_layout()
plt.show()

---
<a id='sec4'></a>
## 4. Ergodicity and the Resolvent Operator

The **resolvent operator** is the geometrically weighted average
$$
\mathbb{M} f(x) = (1-\lambda) \sum_{j=0}^\infty \lambda^j \mathbb{T}^j f(x),
\qquad 0 < \lambda < 1.
$$
In matrix form this is $(1-\lambda)(I - \lambda \mathbb{P})^{-1}\mathbf{f}$.

The Markov process is **ergodic** with respect to $Q$ if and only if for every
nonnegative $\mathbf{f}$ with $\mathbf{q}'\mathbf{f} > 0$, $\mathbb{M}\mathbf{f}(x) > 0$
for all $x$ with $Q$-probability one (Proposition 2.3).

We illustrate this using MC3: starting from only the ergodic block ($\pi=1$),
the process is ergodic; starting from the mixed distribution ($\pi = 0.5$), it is not.

In [ ]:
def resolvent(P, lam=0.9):
    """Compute the resolvent matrix M = (1-lam) * (I - lam*P)^{-1}."""
    n = P.shape[0]
    return (1 - lam) * np.linalg.inv(np.eye(n) - lam * P)

M_mc3 = resolvent(P_mc3, lam=0.9)

# Test vector: indicator of state 3 (the absorbing state)
f_ind = np.array([0., 0., 1.])
Mf = M_mc3 @ f_ind
print("MC3 resolvent applied to indicator of state 3:")
print(f"  Mf = {np.round(Mf, 6)}")
print()
print("  Mf[0] = Mf[1] = 0 => states 1,2 never reach state 3 => NOT ergodic under pi=1")
print("  Mf[2] > 0  => state 3 visits itself => ergodic under pi=0")

# Test vector: indicator of state 1 (first ergodic-block state)
f_ind2 = np.array([1., 0., 0.])
Mf2 = M_mc3 @ f_ind2
print()
print("MC3 resolvent applied to indicator of state 1:")
print(f"  Mf = {np.round(Mf2, 6)}")
print("  Mf[2] = 0 => absorbing state never visits state 1 => confirms non-ergodicity under pi in (0,1)")

---
<a id='sec5'></a>
## 5. Periodicity and Skip-Sampling

The **periodicity** of an irreducible Markov process with respect to $Q$ is the smallest
positive integer $p$ such that $\mathbb{T}^p f = f$ has a non-constant solution.
When no such $p$ exists the process is **aperiodic**.

MC1 has period 2: $\mathbb{P}^2 = I$, so every vector is a fixed point of $\mathbb{T}^2$.
MC3 (ergodic block) is aperiodic.

The **skip-$p$ resolvent**
$$
\mathbb{M}_p \mathbf{f} = (1-\lambda)\sum_{j=0}^\infty \lambda^j \mathbb{P}^{pj} \mathbf{f}
= (1-\lambda)(I - \lambda \mathbb{P}^p)^{-1} \mathbf{f}
$$
being strictly positive for all nonnegative $\mathbf{f}$ with positive $\mathbf{q}'\mathbf{f}$
and all $p \ge 1$ characterizes aperiodicity.

In [ ]:
def skip_resolvent(P, p, lam=0.9):
    """Compute the skip-p resolvent M_p = (1-lam)*(I - lam*P^p)^{-1}."""
    Pp = np.linalg.matrix_power(P, p)
    n = P.shape[0]
    return (1 - lam) * np.linalg.inv(np.eye(n) - lam * Pp)

# MC1: verify P^2 = I
print("MC1: P^2 =")
print(np.linalg.matrix_power(P_mc1, 2))

# Compare eigenvalues of P^j for MC1 and for P_mc3's ergodic 2x2 block
P_ergodic = P_mc3[:2, :2]  # will NOT be stochastic alone, but illustrates spectral structure

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, P_plot, title in [
        (axes[0], P_mc1,    'MC1 (periodic, period 2)'),
        (axes[1], P_mc3,    'MC3 (contains aperiodic block)')]:
    eigenvalues = np.linalg.eigvals(P_plot)
    theta = np.linspace(0, 2 * np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', linewidth=0.7, label='Unit circle')
    ax.scatter(eigenvalues.real, eigenvalues.imag, zorder=5, s=80, label='Eigenvalues')
    ax.set_aspect('equal')
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')

plt.suptitle('Eigenvalues of transition matrices on the complex plane', fontsize=12)
plt.tight_layout()
plt.show()

print("\nMC1 eigenvalues:", np.round(np.linalg.eigvals(P_mc1), 4))
print("MC3 eigenvalues:", np.round(np.linalg.eigvals(P_mc3), 4))

---
<a id='sec6'></a>
## 6. Strong Contraction and Spectral Radius

$\mathbb{T}$ is a **strong contraction** on $\mathcal{N} = \{f : \mathbf{q}'\mathbf{f} = 0\}$
when its spectral radius restricted to $\mathcal{N}$ is strictly less than one.
By Gelfand's theorem, this equals the second-largest eigenvalue modulus of $\mathbb{P}$.

This property governs how quickly the $j$-step conditional expectation
$\mathbb{T}^j f$ converges to its stationary mean $\mathbf{q}'\mathbf{f}$
as $j \to \infty$.

In [ ]:
def spectral_radius_restricted(P, q):
    """
    Spectral radius of T restricted to N = {f : q'f = 0}.
    This equals the second-largest eigenvalue modulus of P.
    """
    eigenvalues = np.linalg.eigvals(P)
    moduli = np.abs(eigenvalues)
    moduli_sorted = np.sort(moduli)[::-1]
    return moduli_sorted[1]  # second largest

# Use MC3 restricted to the ergodic block (states 0 and 1)
P_erg = P_mc3[:2, :2]  # sub-matrix for the ergodic block
# Renormalize rows to make it a valid stochastic matrix
P_erg = P_erg / P_erg.sum(axis=1, keepdims=True)
q_erg = stationary_distributions(P_erg)[0]

rho = spectral_radius_restricted(P_erg, q_erg)
print(f"Ergodic block (MC3, states 1-2): spectral radius of T|_N = {rho:.6f}")
print(f"  => T^m is a strong contraction for any m >= 1")

# Illustrate decay of ||T^j f ||_Q for a centered f
f0 = np.array([1., -1.])          # centered: q'f = 0
norms = []
fj = f0.copy()
for j in range(50):
    norm_j = np.sqrt(q_erg @ (fj ** 2))
    norms.append(norm_j)
    fj = P_erg @ fj

j_vals = np.arange(50)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(j_vals, norms, label=r'$\|\mathbb{T}^j f\|_Q$')
ax.semilogy(j_vals, norms[0] * rho ** j_vals, 'r--', label=rf'$\|f\|_Q \cdot \rho^j$, $\rho={rho:.3f}$')
ax.set_xlabel('$j$')
ax.set_ylabel(r'$\|\mathbb{T}^j f\|_Q$  (log scale)')
ax.set_title('Decay of the $\\mathcal{L}^2$ norm under repeated application of $\\mathbb{T}$')
ax.legend()
plt.tight_layout()
plt.show()

---
<a id='sec7'></a>
## 7. Convergence of Multi-Period Forecasts (Stochastic Stability)

For an aperiodic ergodic process, the $j$-step conditional expectation satisfies
$$
\lim_{j \to \infty} \mathbb{T}^j f(x) = \int f(x)\, Q(dx) \quad \text{for every bounded } f.
$$
That is, long-run forecasts become independent of the current state and equal the
stationary mean.  We illustrate this convergence and the *rate* at which it occurs.

In [ ]:
# Use the ergodic block of MC3
f_test = np.array([1.0, 0.0])           # indicator of state 0
stationary_mean = q_erg @ f_test

forecasts = np.empty((50, 2))           # forecasts[j, x] = T^j f(x)
fj = f_test.copy()
for j in range(50):
    forecasts[j] = fj
    fj = P_erg @ fj

j_vals = np.arange(50)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(j_vals, forecasts[:, 0], label='$\\mathbb{T}^j f(x=0)$')
ax.plot(j_vals, forecasts[:, 1], label='$\\mathbb{T}^j f(x=1)$')
ax.axhline(stationary_mean, color='k', linestyle='--',
           label=rf'Stationary mean $= {stationary_mean:.3f}$')
ax.set_xlabel('Horizon $j$')
ax.set_ylabel(r'$\mathbb{T}^j f(x)$')
ax.set_title('Convergence of multi-period forecasts to the stationary mean')
ax.legend()
plt.tight_layout()
plt.show()

---
<a id='sec8'></a>
## 8. Vector Autoregressions: Ergodic and Non-Ergodic

### 8a. Ergodic VAR

A stable matrix $\mathbb{A}$ (all eigenvalues inside the unit circle) gives an ergodic VAR
$$
X_{t+1} = \mathbb{A} X_t + \mathbb{B} W_{t+1}.
$$
The stationary distribution is $\mathcal{N}(0, \Sigma)$ where $\Sigma$ solves the
**discrete Lyapunov equation** $\Sigma = \mathbb{A}\Sigma\mathbb{A}' + \mathbb{B}\mathbb{B}'$.

In [ ]:
# --- Ergodic VAR ---
A = np.array([[0.9, 0.2],
              [-0.1, 0.7]])
B = np.array([[0.5, 0.0],
              [0.0, 0.3]])

print("Eigenvalues of A:", np.round(np.linalg.eigvals(A), 4))
print("All inside unit circle?", np.all(np.abs(np.linalg.eigvals(A)) < 1))

# Solve discrete Lyapunov equation
Sigma = solve_discrete_lyapunov(A, B @ B.T)
print("\nStationary covariance Sigma (Lyapunov solution):")
print(np.round(Sigma, 4))

# Simulate to verify
T = 5000
X = np.zeros((T, 2))
X[0] = rng.multivariate_normal(np.zeros(2), Sigma)
for t in range(T - 1):
    X[t + 1] = A @ X[t] + B @ rng.standard_normal(2)

Sigma_sample = np.cov(X.T)
print("\nSample covariance (T=5000):")
print(np.round(Sigma_sample, 4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(X[:300, 0], linewidth=0.7, label='$X_{1,t}$')
axes[0].plot(X[:300, 1], linewidth=0.7, label='$X_{2,t}$')
axes[0].set_xlabel('Time $t$')
axes[0].set_title('Ergodic VAR: sample path (first 300 periods)')
axes[0].legend()

axes[1].scatter(X[::5, 0], X[::5, 1], alpha=0.2, s=5)
axes[1].set_xlabel('$X_{1,t}$')
axes[1].set_ylabel('$X_{2,t}$')
axes[1].set_title('Phase portrait (scatter)')
plt.tight_layout()
plt.show()

In [ ]:
# Show convergence of Sigma_t to Sigma as t -> infty (starting from Sigma_0 = 0)
Sigma_t = np.zeros((2, 2))
errors = []
for j in range(150):
    errors.append(np.linalg.norm(Sigma_t - Sigma, 'fro'))
    Sigma_t = A @ Sigma_t @ A.T + B @ B.T

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogy(errors)
ax.set_xlabel('Iteration $t$')
ax.set_ylabel(r'$\|\Sigma_t - \Sigma\|_F$ (log scale)')
ax.set_title('Convergence of $\\Sigma_t$ to the stationary covariance $\\Sigma$')
plt.tight_layout()
plt.show()

### 8b. A Stationary VAR that is Not Ergodic

When the bottom-right block of $\mathbb{A}$ equals 1 and the bottom row of $\mathbb{B}$
is zero, the last state component $X_{2,0}$ is **invariant over time**.
The Law of Large Numbers conditions on $X_{2,0}$, so the process is ergodic
only when $X_{2,0}$ is degenerate.  Conditioning on $X_{2,0}$ identifies it as
an unknown parameter that can be estimated from the time series of $X_{1,t}$.

In [ ]:
# --- Non-ergodic VAR with invariant second component ---
A11 = np.array([[0.8]])         # stable 1x1 block
A12 = np.array([[1.0]])         # coupling
B1  = np.array([[0.4]])

A_ne = np.array([[0.8, 1.0],
                  [0.0, 1.0]])
B_ne = np.array([[0.4],
                  [0.0]])

T_ne = 300
n_draws = 4
X20_values = [-2.0, -0.5, 0.5, 2.0]   # four different realizations of X_{2,0}

mu1_analytic = lambda x20: np.linalg.solve(np.eye(1) - A11, A12 * x20).item()

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
colors = plt.cm.tab10(np.linspace(0, 0.4, n_draws))

running_means = {x20: [] for x20 in X20_values}

for x20, color in zip(X20_values, colors):
    X_path = np.zeros((T_ne, 2))
    X_path[0] = [0.0, x20]
    for t in range(T_ne - 1):
        X_path[t + 1] = A_ne @ X_path[t] + B_ne @ rng.standard_normal(1)
    axes[0].plot(X_path[:, 0], color=color, linewidth=0.8,
                 label=f'$X_{{2,0}}={x20}$')
    cum_mean = np.cumsum(X_path[:, 0]) / np.arange(1, T_ne + 1)
    axes[1].plot(cum_mean, color=color, linewidth=0.8)
    mu_true = mu1_analytic(x20)
    axes[1].axhline(mu_true, color=color, linestyle='--', linewidth=0.8)

axes[0].set_ylabel('$X_{1,t}$')
axes[0].set_title('Non-ergodic VAR: $X_{1,t}$ paths for four values of the invariant $X_{2,0}$')
axes[0].legend(fontsize=9)
axes[1].set_ylabel('Running mean of $X_{1,t}$')
axes[1].set_xlabel('Time $t$')
axes[1].set_title('Running mean converges to a limit that depends on $X_{2,0}$ (dashed = analytic limit)')
plt.tight_layout()
plt.show()

---
<a id='sec9'></a>
## 9. Inventing a Past: Moving-Average Representation

For a stable VAR, drawing the initial condition from $\mathcal{N}(0, \Sigma_\infty)$
is equivalent to having run the process from $t = -\infty$:
$$
X_t = \sum_{j=0}^\infty \mathbb{A}^j \mathbb{B}\, W_{t-j}.
$$
We verify this by comparing the empirical covariance of the truncated
moving-average to the Lyapunov solution $\Sigma_\infty$, and by
showing that increasing the truncation depth $J$ drives the approximation error to zero.

In [ ]:
# Moving-average simulation: X_t = sum_{j=0}^{J} A^j B W_{t-j}
T_ma   = 3000
J_list = [5, 20, 50, 100]

# Pre-generate a long white-noise sequence
W_long = rng.standard_normal((T_ma + max(J_list), 2))

errors_ma = []
for J in J_list:
    # MA coefficients C_j = A^j B,  j = 0,...,J
    C = [np.linalg.matrix_power(A, j) @ B for j in range(J + 1)]

    # Build X_t
    X_ma = np.zeros((T_ma, 2))
    for t in range(T_ma):
        base = t + J   # index into W_long
        X_ma[t] = sum(C[j] @ W_long[base - j] for j in range(J + 1))

    Sigma_sample_ma = np.cov(X_ma.T)
    errors_ma.append(np.linalg.norm(Sigma_sample_ma - Sigma, 'fro'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(J_list, errors_ma, 'o-')
axes[0].set_xlabel('Truncation depth $J$')
axes[0].set_ylabel(r'$\|\hat{\Sigma} - \Sigma_\infty\|_F$')
axes[0].set_title('Approximation error of truncated MA vs Lyapunov $\\Sigma_\\infty$')

# Use J=100 path and compare with direct simulation from stationary distribution
J_best = 100
C_best = [np.linalg.matrix_power(A, j) @ B for j in range(J_best + 1)]
X_ma_best = np.zeros((T_ma, 2))
W_best = rng.standard_normal((T_ma + J_best, 2))
for t in range(T_ma):
    base = t + J_best
    X_ma_best[t] = sum(C_best[j] @ W_best[base - j] for j in range(J_best + 1))

axes[1].scatter(X_ma_best[::5, 0], X_ma_best[::5, 1], alpha=0.15, s=4,
                label=f'MA (J={J_best})')
axes[1].set_xlabel('$X_{1,t}$')
axes[1].set_ylabel('$X_{2,t}$')
axes[1].set_title('Stationary phase portrait via moving-average representation')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Lyapunov Sigma:")
print(np.round(Sigma, 4))
print(f"\nSample covariance from MA(J={J_best}):")
print(np.round(np.cov(X_ma_best.T), 4))

---
## Summary

| Section | Key concept illustrated |
|---------|------------------------|
| 1 | Transition matrix $\mathbb{P}$; sample paths of three chains |
| 2 | Stationary distributions as left unit eigenvectors; convex hull |
| 3 | Right unit eigenvectors = invariant functions; $f(X_t)$ constant along paths |
| 4 | Resolvent operator $\mathbb{M}$; positivity as criterion for ergodicity |
| 5 | Periodicity; eigenvalue portrait; skip-$p$ resolvent |
| 6 | Spectral radius of $\mathbb{T}|_{\mathcal{N}}$; exponential decay of $\|\mathbb{T}^j f\|_Q$ |
| 7 | Convergence of $\mathbb{T}^j f$ to stationary mean |
| 8a | Ergodic VAR; discrete Lyapunov equation; convergence of $\Sigma_t$ |
| 8b | Non-ergodic VAR; running mean depends on invariant $X_{2,0}$ |
| 9 | Moving-average representation; truncation error vs $\Sigma_\infty$ |